# Sentinel-1-only LULC: training and evaluation
Run top to bottom. Export first (README step 2); this notebook assumes `data/raw` on Drive. Labels are Dynamic World monthly maps, so the monthly outputs have **no independent ground truth**; WorldCover and Esri are the independent checks.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/sar-lulc
!pip -q install -r requirements.txt
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
DATA='/content/drive/MyDrive/lulc/raw'; RUNS='/content/drive/MyDrive/lulc/runs'

## 1. V0 and V1 (single month, scratch vs ImageNet), 3 seeds each

In [ ]:
%run scripts/train_all.py --data $DATA --out $RUNS --variants V0_scratch_1m V1_imagenet_1m

## 2. V2-V4 (3-month window, confidence cutoff, coherence). Encoder weights come from V0 vs V1.

In [ ]:
%run scripts/train_all.py --data $DATA --out $RUNS --variants V2_window3 V3_window3_conf50 V4_window3_coh50

## 3. Training curves and variant table
Gaps below 0.05 mIoU are inside seed noise.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
fig,ax=plt.subplots(1,2,figsize=(11,4))
for d in sorted(glob.glob(f'{RUNS}/V*/seed*')):
    h=pd.read_csv(f'{d}/history.csv'); v=d.split('/')[-2]
    ax[0].plot(h.epoch,h.train_loss,alpha=.5,label=v if d.endswith('seed0') else None)
    ax[1].plot(h.epoch,h.val_miou,alpha=.5,label=v if d.endswith('seed0') else None)
ax[0].set_title('train loss'); ax[1].set_title('validation mIoU (EMA weights)')
for a in ax: a.set_xlabel('epoch'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{RUNS}/curves.png',dpi=150); plt.show()
pd.read_csv(f'{RUNS}/val_summary.csv').round(3)

## 4. Test evaluation (choose the variant by validation, not by test)

In [ ]:
BEST='V3_window3_conf50'   # set from the validation table above
%run scripts/evaluate.py --data $DATA --runs $RUNS --variant $BEST

In [ ]:
import json; m=json.load(open(f'{RUNS}/eval_{BEST}/metrics.json'))
for k,v in m.items(): print(k, {c:round(x,3) for c,x in v['per_class_iou'].items()}, 'F1', {c:round(x,3) for c,x in v['per_class_f1'].items()})
pd.read_csv(f'{RUNS}/eval_{BEST}/per_city.csv').round(3)

## 5. False-change test and out-of-year test
`change.csv`: change reported on pixels where WorldCover and Esri agree (should be stable). Out-of-year: export test cities for 2024 (`--year 2024`) and score the 2021-trained model.

In [ ]:
pd.read_csv(f'{RUNS}/eval_{BEST}/change.csv').groupby('maps')[['flip_rate_pct','monthly_churn_pp','net_total_change_pp']].mean().round(2)

In [ ]:
# After exporting 2024 for the test cities into DATA24:
# DATA24='/content/drive/MyDrive/lulc/raw2024'
# %run scripts/evaluate.py --data $DATA24 --runs $RUNS --variant $BEST --year 2024 --splits test --tag _2024